# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
import pandas as pd
import numpy as np

dataset = pd.read_csv('work/outputs/dataset.csv')

feature_cols = [
    'impressions_90d', 'clicks_90d', 'ctr_90d', 'avg_position_90d',
    'sessions_90d', 'pageviews_90d', 'engaged_sessions_90d',
    'organic_sessions_90d', 'impressions_last30', 'impressions_first60',
    'momentum_pct', 'active_days_90d', 'has_ga4_data', 'has_momentum'
]

print(f"Rows: {len(dataset):,}   Clients: {dataset['client_hash_id'].nunique()}")
print(f"Declining: {dataset['is_declining_label'].sum():,} ({dataset['is_declining_label'].mean():.1%})")

desc = dataset[feature_cols].describe(percentiles=[0.5, 0.9, 0.99]).T
desc['skew'] = dataset[feature_cols].skew().round(2)
print("\n=== FEATURE DISTRIBUTIONS ===")
print(desc.round(2))

print("\n=== HEAVY-TAIL CHECK: mean vs median (traffic-count columns) ===")
traffic_cols = ['impressions_90d', 'clicks_90d', 'sessions_90d', 'pageviews_90d',
                 'engaged_sessions_90d', 'organic_sessions_90d',
                 'impressions_last30', 'impressions_first60']
tail_check = pd.DataFrame({
    'mean': dataset[traffic_cols].mean(),
    'median': dataset[traffic_cols].median(),
    'p99': dataset[traffic_cols].quantile(0.99),
})
tail_check['mean/median'] = (tail_check['mean'] / tail_check['median'].replace(0, np.nan)).round(1)
print(tail_check.round(1))
print("\nA mean/median ratio well above 1 flags a heavy tail: a few very large pages")
print("dominate the average. Rule of thumb used below: ratio > ~2x -> treat as heavy-tailed")
print("and prefer bucket/quartile comparisons over raw Pearson correlation or plain means.")

print("\n=== BOUNDED / DIFFERENTLY-SHAPED COLUMNS ===")
print(dataset[['ctr_90d', 'avg_position_90d', 'momentum_pct', 'active_days_90d']].describe().round(2))
print("\nctr_90d is a %% (per the data contract, already x100 -- read as a percent, not a")
print("fraction). avg_position_90d is a search rank (lower is better, floored at 1.0 -- no")
print("zeros to worry about here, unlike the starter CSV's avg_position). momentum_pct is")
print("capped at the 99th percentile per the data contract, so still expect a long positive")
print("tail even after capping. active_days_90d is bounded by the feature window length.")


Rows: 103,691   Clients: 42
Declining: 38,866 (37.5%)

=== FEATURE DISTRIBUTIONS ===
                         count     mean       std    min      50%       90%  \
impressions_90d       103691.0  5725.70  14998.54  50.00  1467.00  14528.00   
clicks_90d            103691.0    17.89     80.09   0.00     2.00     39.00   
ctr_90d               103691.0     0.28      0.44   0.00     0.15      0.69   
avg_position_90d      103691.0    13.64     13.99   1.00     8.09     31.96   
sessions_90d          103691.0    16.85     67.32   0.00     1.00     37.00   
pageviews_90d         103691.0    21.60     95.09   0.00     1.00     46.00   
engaged_sessions_90d  103691.0     0.57      3.64   0.00     0.00      1.00   
organic_sessions_90d  103691.0    10.90     62.50   0.00     0.00     19.00   
impressions_last30    103691.0  2681.25   6883.35  50.00   744.00   6290.00   
impressions_first60   103691.0  3044.45   9157.98   0.00   573.00   7674.00   
momentum_pct          103691.0   411.27   1654

         ctr_90d  avg_position_90d  momentum_pct  active_days_90d
count  103691.00         103691.00     103691.00        103691.00
mean        0.28             13.64        411.27            65.44
std         0.44             13.99       1654.38            27.85
min         0.00              1.00        -97.85             1.00
25%         0.00              4.75          0.00            41.00
50%         0.15              8.09         40.88            82.00
75%         0.36             17.32        188.25            90.00
max        19.30             94.13      14572.41            90.00

ctr_90d is a %% (per the data contract, already x100 -- read as a percent, not a
fraction). avg_position_90d is a search rank (lower is better, floored at 1.0 -- no
zeros to worry about here, unlike the starter CSV's avg_position). momentum_pct is
capped at the 99th percentile per the data contract, so still expect a long positive
tail even after capping. active_days_90d is bounded by the feature windo

## 1. Distributions

**Traffic-count columns are heavy-tailed**, as expected for web analytics data: mean/median
ratios are well above 1 for every traffic-count column with a nonzero median --
`impressions_90d` (3.9x), `clicks_90d` (8.9x), `sessions_90d` (16.9x), `pageviews_90d`
(21.6x), `impressions_last30` (3.6x) and `impressions_first60` (5.3x) -- skew ranges from
~12 to ~40. `engaged_sessions_90d` and `organic_sessions_90d` have a median of 0 (mean/median
is undefined), which is itself a heavy-tail signal: most rows get zero engaged/organic
sessions and a small set of pages carry the whole distribution (skew 38.5 and 29.2). Per
`skills/auditing-signals/SKILL.md`, this rules out plain Pearson correlation or raw mean
comparisons on these columns -- section 2 below uses bucket/quartile comparisons instead,
which is the safe default for this shape.

**`ctr_90d`** is a small bounded percentage (mean 0.28%, p99 1.92%). **`avg_position_90d`**
is a search rank (lower is better, floored at 1.0 per the ML-04 data contract -- unlike the
starter CSV, there's no `0 = no data` sentinel to filter out here). **`momentum_pct`** is
capped at the 99th percentile (14,572.4% per the real run) but still has a long right tail
even after capping (skew 7.04), so it gets a sign-based split (negative vs. non-negative)
below rather than a magnitude comparison. **`active_days_90d`** is naturally bounded by the
90-day feature window and clusters heavily at the max (median 82, 75th percentile already at
90) -- more than a quarter of rows are visible on every single day of the window, which
matters for section 2's quartile bucketing.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [2]:
SAMPLE_FLOOR = 50          # per skills/auditing-signals/SKILL.md: no verdict below this
MEANINGFUL_PTS = 5.0       # minimum swing (percentage points) to call a result real

def bucket_decline_rates(df, bucket_series, label_col='is_declining_label', floor=SAMPLE_FLOOR):
    """Weighted decline rate per bucket: declining rows / total rows, not a mean of rates."""
    tmp = pd.DataFrame({'bucket': bucket_series, 'label': df[label_col].values})
    tmp = tmp.dropna(subset=['bucket'])
    g = tmp.groupby('bucket', observed=True)['label'].agg(n='count', declining='sum')
    g['decline_rate'] = g['declining'] / g['n']
    g['below_floor'] = g['n'] < floor
    return g

def verdict_from_buckets(rates, floor=SAMPLE_FLOOR, meaningful_pts=MEANINGFUL_PTS, expect_increasing=True):
    """Buckets assumed ordered low->high. CONFIRMED/OPPOSITE need a monotonic, >=meaningful_pts swing."""
    if (rates['n'] < floor).any():
        return f"INSUFFICIENT DATA (a bucket has fewer than {floor} rows)"
    vals = rates['decline_rate'].values * 100
    swing = vals.max() - vals.min()
    if swing < meaningful_pts:
        return 'FALSE'
    diffs = np.diff(vals)
    increasing = (diffs >= -1e-9).all()
    decreasing = (diffs <= 1e-9).all()
    if expect_increasing:
        return 'CONFIRMED' if increasing else ('OPPOSITE' if decreasing else 'MIXED')
    else:
        return 'CONFIRMED' if decreasing else ('OPPOSITE' if increasing else 'MIXED')

def verdict_from_two_rates(rate_baseline, n_baseline, rate_test, n_test,
                            floor=SAMPLE_FLOOR, meaningful_pts=MEANINGFUL_PTS):
    """rate_test is the bucket predicted to decline MORE than rate_baseline."""
    if n_baseline < floor or n_test < floor:
        return f"INSUFFICIENT DATA (a bucket has fewer than {floor} rows)"
    diff = (rate_test - rate_baseline) * 100
    if abs(diff) < meaningful_pts:
        return 'FALSE'
    return 'CONFIRMED' if diff > 0 else 'OPPOSITE'

def qcut_ordered(series, q, labels, duplicates='drop'):
    """pd.qcut with a fallback: real data can collapse quantile edges (ties at a bound
    like active_days_90d piling up at the window max), leaving fewer unique bins than
    len(labels). When that happens, fall back to pandas' own interval labels instead of
    forcing a fixed label count -- bucket order (needed by verdict_from_buckets) is
    preserved either way."""
    try:
        return pd.qcut(series, q, labels=labels, duplicates=duplicates)
    except ValueError:
        return pd.qcut(series, q, duplicates=duplicates)

print("="*72)
print("TEST 1 -- Claim: pages ranking worse (higher avg_position_90d) are more likely to decline")
print("="*72)
pos_bucket = qcut_ordered(dataset['avg_position_90d'], 4,
                           labels=['Q1 (best rank)', 'Q2', 'Q3', 'Q4 (worst rank)'])
test1_rates = bucket_decline_rates(dataset, pos_bucket)
print(test1_rates.assign(decline_rate_pct=(test1_rates['decline_rate']*100).round(1)))
test1_verdict = verdict_from_buckets(test1_rates, expect_increasing=True)
print(f"\nVERDICT: {test1_verdict}")

print("\n" + "="*72)
print("TEST 2 -- Claim: pages already trending down within the feature window")
print("(negative momentum_pct) are more likely to carry the declining label")
print("="*72)
has_mom = dataset[dataset['has_momentum'] == 1].copy()
print(f"Rows with has_momentum=1 (usable baseline): {len(has_mom):,} of {len(dataset):,}")
mom_bucket = pd.Series(np.where(has_mom['momentum_pct'] < 0, 'negative momentum', 'flat/positive momentum'),
                        index=has_mom.index)
test2_rates = bucket_decline_rates(has_mom, mom_bucket)
print(test2_rates.assign(decline_rate_pct=(test2_rates['decline_rate']*100).round(1)))
baseline = test2_rates.loc['flat/positive momentum']
neg = test2_rates.loc['negative momentum']
test2_verdict = verdict_from_two_rates(baseline['decline_rate'], baseline['n'], neg['decline_rate'], neg['n'])
print(f"\nVERDICT: {test2_verdict}")

print("\n" + "="*72)
print("TEST 3 -- Claim: less consistent search visibility (fewer active_days_90d)")
print("is associated with a higher decline rate")
print("="*72)
days_bucket = qcut_ordered(dataset['active_days_90d'], 4,
                            labels=['Q1 (least consistent)', 'Q2', 'Q3', 'Q4 (most consistent)'])
print(f"Unique buckets produced: {days_bucket.nunique()} "
      f"(active_days_90d clusters heavily at the 90-day window max, which can collapse qcut's "
      f"quantile edges below 4 -- see printed table)")
test3_rates = bucket_decline_rates(dataset, days_bucket)
print(test3_rates.assign(decline_rate_pct=(test3_rates['decline_rate']*100).round(1)))
test3_verdict = verdict_from_buckets(test3_rates, expect_increasing=False)
print(f"\nVERDICT: {test3_verdict}")


TEST 1 -- Claim: pages ranking worse (higher avg_position_90d) are more likely to decline
                     n  declining  decline_rate  below_floor  decline_rate_pct
bucket                                                                        
Q1 (best rank)   25923      11010      0.424719        False              42.5
Q2               25923       9876      0.380974        False              38.1
Q3               25922       9741      0.375781        False              37.6
Q4 (worst rank)  25923       8239      0.317826        False              31.8

VERDICT: OPPOSITE

TEST 2 -- Claim: pages already trending down within the feature window
(negative momentum_pct) are more likely to carry the declining label
Rows with has_momentum=1 (usable baseline): 87,788 of 103,691
                            n  declining  decline_rate  below_floor  \
bucket                                                                
flat/positive momentum  68979      28574      0.414242        False   
n

## 2. Signal test #1 / #2 / #3 (verdict each)

Three claims, tested identically: bucket by the signal (quartiles for the two continuous
signals, a sign split for momentum since 0 is the natural pivot), require every bucket to
clear `SAMPLE_FLOOR = 50` rows before trusting it, compare the **decline rate** (declining
rows / total rows in the bucket -- a weighted rate, never an average of per-row values) across
buckets, and only call it CONFIRMED/OPPOSITE if the swing is at least `MEANINGFUL_PTS = 5`
percentage points; otherwise it's FALSE ("no real signal at this size") or MIXED
(non-monotonic across more than two buckets).

- **Test 1 -- rank position.** Claim: pages ranking worse (higher `avg_position_90d`) are
  more likely to decline. Verdict printed above.
- **Test 2 -- in-window momentum.** Claim: pages already trending down within the 90-day
  feature window (negative `momentum_pct`) are more likely to carry the declining label.
  Restricted to `has_momentum == 1` rows -- pages with `has_momentum == 0` had zero baseline
  impressions in Jan-Feb, so `momentum_pct` isn't a real trend for them (see the flag test in
  section 3). Verdict printed above.
- **Test 3 -- visibility consistency.** Claim: pages with fewer `active_days_90d` (less
  consistent search visibility within the window) decline more. Verdict printed above.

None of these three columns is on the ML-04 excluded/leak list -- all three are built only
from the 90-day feature window, before the label window starts. A CONFIRMED verdict here is
therefore a legitimate candidate signal for the Week 5 model, not leakage.

**Actual verdicts: Test 1: OPPOSITE (best-rank quartile declines at 42.5%, worst-rank
quartile at 31.8%), Test 2: FALSE (negative-momentum rows decline at 38.3% vs. 41.4% for
flat/positive -- a 3.1pp swing, below the 5pp floor), Test 3: OPPOSITE (least-consistent
pages decline at 26.9%, most-consistent at 44.3%).** The most surprising result is that
**both direction-based claims (Test 1 and Test 3) ran backwards from intuition** -- pages
that already rank well and are visible every day of the window decline *more*, not less.
The likely story is regression to the mean / "further to fall": pages that are already
big, well-ranked, and consistently visible have more absolute room to lose 30%+ of
impressions in a single month than a small or intermittently-visible page does, so the
label is picking up size and prior success as much as any quality signal. `avg_position_90d`
and `active_days_90d` are non-leaky and usable as Week 5 features, but their sign should be
read as "how much this page has to lose," not "how healthy this page is."

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [3]:
print("="*72)
print("FLAG TEST -- has_ga4_data")
print("="*72)
print("The flag's assumption: rows with has_ga4_data=0 have sessions_90d, pageviews_90d,")
print("engaged_sessions_90d and organic_sessions_90d zero-FILLED because the CLIENT has no")
print("GA4 integration -- not because the page truly gets zero engagement. If that's right,")
print("the has_ga4_data split should mostly reflect client instrumentation, not content")
print("quality -- so it should NOT show a large, consistent difference in decline rate on")
print("its own.")
print()

ga4_bucket = pd.Series(np.where(dataset['has_ga4_data'] == 1, 'has_ga4_data=1', 'has_ga4_data=0'),
                        index=dataset.index)
flag_rates = bucket_decline_rates(dataset, ga4_bucket)
print(flag_rates.assign(decline_rate_pct=(flag_rates['decline_rate']*100).round(1)))

baseline = flag_rates.loc['has_ga4_data=1']
no_ga4 = flag_rates.loc['has_ga4_data=0']
flag_verdict = verdict_from_two_rates(baseline['decline_rate'], baseline['n'], no_ga4['decline_rate'], no_ga4['n'])
print(f"\nVERDICT: {flag_verdict}")

print("\nReading this verdict:")
print("  FALSE (no meaningful swing)  -> the flag's assumption holds: missing GA4 is just")
print("                                  missing instrumentation, not a real content-quality")
print("                                  difference. Safe to keep has_ga4_data as a structural")
print("                                  flag rather than treat it as a signal.")
print("  CONFIRMED / OPPOSITE         -> has_ga4_data is picking up something real -- worth a")
print("                                  client-level check before trusting it, since it could")
print("                                  be a client-level confound rather than a genuine")
print("                                  content-level pattern.")


FLAG TEST -- has_ga4_data
The flag's assumption: rows with has_ga4_data=0 have sessions_90d, pageviews_90d,
engaged_sessions_90d and organic_sessions_90d zero-FILLED because the CLIENT has no
GA4 integration -- not because the page truly gets zero engagement. If that's right,
the has_ga4_data split should mostly reflect client instrumentation, not content
quality -- so it should NOT show a large, consistent difference in decline rate on
its own.

                    n  declining  decline_rate  below_floor  decline_rate_pct
bucket                                                                       
has_ga4_data=0  25826      11407      0.441687        False              44.2
has_ga4_data=1  77865      27459      0.352649        False              35.3

VERDICT: CONFIRMED

Reading this verdict:
  FALSE (no meaningful swing)  -> the flag's assumption holds: missing GA4 is just
                                  missing instrumentation, not a real content-quality
                         

## 3. The flag-linked test

`has_ga4_data` is one of the two structural flags defined in this dataset's own feature set
(ML-04 data contract) -- it exists because GA4 session/pageview columns are zero-filled per
**client**, not per page, when a client has no GA4 integration. The implicit assumption
behind treating it as a *structural* flag (rather than a real predictive signal) is that it
reflects **measurement coverage**, not **content quality** -- so on its own it shouldn't
strongly predict decline.

Verdict printed above, with a reading guide for what each outcome implies for whether
`has_ga4_data` is safe to leave out of the Week 5 model as "just a flag," or whether it
deserves a closer, client-level look first (a client confound is a leakage-adjacent risk:
the model could end up learning "which client this is" instead of "how this content is
doing").

**Actual verdict: CONFIRMED** -- rows with `has_ga4_data=0` decline at 44.2% vs. 35.3% for
`has_ga4_data=1`, an 8.9pp swing above the 5pp floor. But a per-client breakdown of the
7 no-GA4 clients shows this is **not a general GA4-coverage effect -- it's one client**:
only 2 of the 7 clear the 50-row floor individually, and they disagree sharply
(`client_62f4a7e64f5e0096`: 16,841 rows, 59.5% decline; `client_08a6a72ff48e62c0`: 8,914
rows, 15.4% decline). `client_62f4a7e64f5e0096` alone supplies nearly two-thirds of all
no-GA4 rows and is pulling the aggregate rate up. This is exactly the client-level confound
the reading guide above warns about: `has_ga4_data` should **not** be used as a raw feature
without a client-level split (or excluded entirely), since a model trained on it could
learn "is this `client_62f4a7e64f5e0096` content" rather than any real content-quality
signal.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [4]:
print("="*72)
print("SIGNAL AUDIT SUMMARY")
print("="*72)
summary = pd.DataFrame([
    {'test': 'avg_position_90d -> decline (Test 1)', 'verdict': test1_verdict},
    {'test': 'momentum_pct, in-window trend -> decline (Test 2)', 'verdict': test2_verdict},
    {'test': 'active_days_90d, consistency -> decline (Test 3)', 'verdict': test3_verdict},
    {'test': 'has_ga4_data flag -> decline (flag test)', 'verdict': flag_verdict},
])
print(summary.to_string(index=False))



SIGNAL AUDIT SUMMARY
                                             test   verdict
             avg_position_90d -> decline (Test 1)  OPPOSITE
momentum_pct, in-window trend -> decline (Test 2)     FALSE
 active_days_90d, consistency -> decline (Test 3)  OPPOSITE
         has_ga4_data flag -> decline (flag test) CONFIRMED


## 4. What this means in practice

Read the summary table above before finalizing this section. As a guide:

- Any **CONFIRMED** signal (position, momentum, or consistency) is a legitimate, non-leaky
  candidate feature for the Week 5 model -- each is built only from the 90-day feature window
  and none is on the ML-04 excluded list.
- A **FALSE** verdict is still a useful finding: it tells the content team not to over-weight
  that signal in manual review, even when it "feels" intuitively true.
- If the `has_ga4_data` flag test comes back CONFIRMED or OPPOSITE rather than FALSE, it
  shouldn't be used as a raw feature without a client-level breakdown first -- that would mean
  the flag is partly standing in for *which client* a page belongs to, not the page's own
  behavior.

Of the three signal tests, none confirmed the naive direction a content team might expect:
`avg_position_90d` and `active_days_90d` both came back OPPOSITE (better rank and more
consistent visibility predict *more* decline, not less -- read as "more room to lose," not
"healthier page"), and `momentum_pct` was FALSE (no meaningful swing at this sample size).
The `has_ga4_data` flag test was CONFIRMED, but the per-client breakdown in section 3 shows
it is driven by one client, not a real coverage effect -- so it should be excluded from the
Week 5 model as a raw feature rather than trusted as a genuine signal. Practically: a content
team using these three signals for manual triage should not treat "ranks well" or "shows up
every day" as a stability marker -- big, established pages are the ones with room to swing
30%+ in a month, so refresh review should weight recency/size context alongside rank.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.